# LOF top10 チャンク詳細観察

**目的**: second pass LOF の上位チャンクが実際にどんなログを含んでいるかを確認する。

- データ: `S3 attack day`, `cu10` (ユーザ単位・10分窓)
- 対象: `phase4_secondpass_lof_rulefree_s3_cu10` の top100 チャンクを seed potential 分析したもの
- 注目チャンク: `chunk203`, `chunk204`, `chunk210` (office-doc), `chunk243`, `chunk244` (user-file)

In [ ]:
import json
import pickle
from pathlib import Path
from collections import Counter

import pandas as pd

ROOT = Path("../")

SEED_RESULTS = ROOT / "analysis_data/model_runs/phase4_secondpass_lof_seed_potential_s3_cu10/results.json"
SESSION_PKL  = ROOT / "analysis_data/atlasv2_for_deep-loglizer/exp_benign1_200k_vs_s3_cu10/session_test.pkl"

with open(SEED_RESULTS, encoding="utf-8") as f:
    seed_data = json.load(f)

with open(SESSION_PKL, "rb") as f:
    sessions = pickle.load(f)

rows = seed_data["rows"]
print(f"ロード完了: チャンク数={len(rows)}, セッション数={len(sessions)}")

## 1. top-k ごとの密度まとめ

In [ ]:
topk_df = pd.DataFrame(seed_data["topk_summary"])
topk_df = topk_df[["topk", "events", "attack_mixed_chunks", "pure_normal_chunks",
                    "useful_seed_chunks", "normal_ratio", "category_counts"]]
topk_df["normal_ratio"] = topk_df["normal_ratio"].map("{:.1%}".format)
display(topk_df)

## 2. top10 チャンク一覧

In [ ]:
top10_records = []
for row in rows[:10]:
    top_procs = ", ".join(p for p, _ in row["top_processes"][:3])
    top_eids  = ", ".join(f"EID{e}({c})" for e, c in row["top_event_ids"][:3])
    chunk_label = f"chunk{row['chunk_index']:03d}"
    top10_records.append({
        "rank": row["rank"],
        "chunk": chunk_label,
        "session": row["parent_session_id"],
        "category": row["category"],
        "score": f"{row['score']:.2e}",
        "attack / normal": f"{row['attack_events']} / {row['normal_events']}",
        "top_processes": top_procs,
        "top_event_ids": top_eids,
        "useful_seed": "★" if row["is_useful_seed"] else "",
    })

top10_df = pd.DataFrame(top10_records)
display(top10_df)

## 3. 生ログ観察ヘルパー

以下のセルでチャンクインデックスを指定すると、そのチャンクの生ログ（templateテキスト）を全行出力する。

In [ ]:
PROCESS_KEYS = ["ProcessName", "NewProcessName", "Image", "Application"]

def parse_template(template: str) -> dict:
    fields = {}
    for part in template.split(" | "):
        if "=" not in part:
            continue
        key, value = part.split("=", 1)
        fields[key] = value
    return fields

def process_name(fields: dict) -> str:
    for key in PROCESS_KEYS:
        v = fields.get(key)
        if v:
            return Path(v).name.lower()
    return "-"

def show_chunk(parent_session_id: str, chunk_index: int, chunk_size: int = 100) -> None:
    if parent_session_id not in sessions:
        print(f"セッション '{parent_session_id}' が見つかりません")
        return

    session = sessions[parent_session_id]
    templates = session["templates"]
    labels    = session.get("labels", [0] * len(templates))

    start = chunk_index * chunk_size
    end   = min(len(templates), start + chunk_size)

    print(f"=== {parent_session_id} | chunk{chunk_index:03d} (event {start}–{end-1}) ===")
    print(f"    総イベント数: {end - start}")
    print()

    proc_counter = Counter()
    eid_counter  = Counter()
    attack_lines = []

    for i, (tmpl, label) in enumerate(zip(templates[start:end], labels[start:end])):
        fields = parse_template(tmpl)
        proc   = process_name(fields)
        eid    = fields.get("EventID", "-")
        proc_counter[proc] += 1
        eid_counter[eid]   += 1
        flag = " [ATTACK]" if label else ""
        print(f"  [{start + i:05d}]{flag} EID={eid:5s} proc={proc:30s} | {tmpl[:120]}")

    print()
    print("プロセス集計:", proc_counter.most_common(8))
    print("EID集計:     ", eid_counter.most_common(8))
    if attack_lines:
        print(f"ATTACK イベント数: {len(attack_lines)}")

print("ヘルパー定義完了")

## 4. rank1–3: attack-mixed チャンク（参考）

In [ ]:
# rank1: chunk330 — payload.exe 中心、攻撃混在
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 330)

In [ ]:
# rank2: chunk308 — tpautoconnect + cmd.exe、攻撃少量
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 308)

In [ ]:
# rank3: chunk325 — payload.exe 中心
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 325)

## 5. rank4–7, 9: useful seed チャンク（純正常）

これらが今日確定したいユースケース候補。

In [ ]:
# rank4: chunk203 — office-doc (winword + repmgr)
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 203)

In [ ]:
# rank5: chunk204 — office-doc (winword + repmgr)
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 204)

In [ ]:
# rank6: chunk244 — user-file (explorer.exe)
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 244)

In [ ]:
# rank7: chunk210 — office-doc (winword)
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 210)

In [ ]:
# rank9: chunk243 — user-file (explorer.exe)
show_chunk("win-32-h1|aalsahee|20220719T1430Z", 243)

## 6. rank8, 10: background-vmware / tpautoconnect（参考）

In [ ]:
# rank8, 10 を確認するにはrows[7], rows[9]からchunk_indexを取得
for row in rows[7:11]:
    if row["rank"] in (8, 10):
        print(f"rank{row['rank']}: chunk{row['chunk_index']:03d} category={row['category']}")
        show_chunk(row["parent_session_id"], row["chunk_index"])
        print()

## 7. 任意チャンクを自由に観察

以下のセルで `chunk_index` を書き換えて実行すると任意のチャンクを確認できる。

In [ ]:
# --- ここを書き換えて使う ---
TARGET_SESSION = "win-32-h1|aalsahee|20220719T1430Z"
TARGET_CHUNK   = 203
# ---------------------------

show_chunk(TARGET_SESSION, TARGET_CHUNK)

## 8. top50 以降の browser 系候補（参考）

rank 42–47 付近に firefox.exe が現れる。

In [ ]:
browser_rows = [r for r in rows if r["category"] == "browser"]
print(f"browser チャンク数: {len(browser_rows)}")
for r in browser_rows:
    top_procs = ", ".join(p for p, _ in r["top_processes"][:3])
    print(f"  rank{r['rank']:3d}: chunk{r['chunk_index']:03d}  {top_procs}")

In [ ]:
# browser チャンクの1つを観察（rank42 付近）
if browser_rows:
    first_browser = browser_rows[0]
    show_chunk(first_browser["parent_session_id"], first_browser["chunk_index"])